# Báo cáo phân tích kết quả thi THPT

## Hai câu hỏi nghiên cứu từ dashboard

Báo cáo chuyển từ việc quan sát các biểu đồ riêng lẻ sang kiểm tra cấu trúc đứng sau những chỉ số tổng hợp. Trọng tâm là xác định liệu một xu hướng chung có thực sự mang tính phổ quát hay đang được hình thành bởi một số thành phần nổi bật.

Hai câu hỏi được lựa chọn gồm:

1. **Sự thay đổi điểm trung bình toàn quốc đến từ toàn bộ môn thi hay chỉ tập trung ở một số môn?**
2. **Chênh lệch vùng miền phản ánh khác biệt chung giữa các địa phương hay chủ yếu bị chi phối bởi một số tỉnh, thành?**

Các câu hỏi đều được trả lời bằng những biểu đồ đã có trong ứng dụng và bằng đúng các chỉ số mà dashboard đang sử dụng. Những con số bổ sung trong notebook chỉ nhằm phân rã, đối chiếu và làm rõ thông tin đã có trên biểu đồ; không thay đổi định nghĩa dữ liệu gốc.


## 1. Phạm vi, nguồn dữ liệu và thiết kế phân tích

### Nguồn dữ liệu

Notebook đọc trực tiếp bộ dữ liệu do dashboard sinh ra tại **frontend/src/data/data.ts**. Cách làm này bảo đảm các kết quả trong báo cáo có thể đối chiếu với số liệu và bộ lọc đang hiển thị trên ứng dụng.

### Phạm vi và biểu đồ sử dụng

| Câu hỏi | Phạm vi dữ liệu | Biểu đồ làm căn cứ |
|---|---|---|
| Câu hỏi 1 | Chương trình CT2006, giai đoạn 2022–2024 | Tab **Tổng quan**: điểm trung bình toàn quốc theo năm; tab **Xu hướng & Môn học**: điểm trung bình theo từng môn |
| Câu hỏi 2 | Năm 2026, môn **Tổng hợp**, phạm vi **Toàn quốc** | Tab **Địa phương & Vùng miền**: điểm trung bình theo vùng, Top/Bottom tỉnh thành và bảng xếp hạng |

### Quy ước tính toán và diễn giải

- Câu hỏi 1 chỉ sử dụng CT2006 trong ba năm 2022–2024. Việc giữ cố định chương trình giúp tách biến động điểm số khỏi sự khác biệt do thay đổi chương trình thi.
- Theo logic của dashboard, điểm trung bình toàn quốc trong mỗi năm là trung bình cộng của các điểm trung bình môn đủ điều kiện trong năm đó.
- Với câu hỏi 2, điểm **Tổng hợp** của một tỉnh là trung bình các điểm trung bình môn của tỉnh; điểm **Tổng hợp** của vùng được tính từ các điểm trung bình môn tương ứng của vùng. Các chỉ số trong notebook tái tạo cùng logic này.
- Khoảng cách nội vùng được đo bằng điểm của tỉnh cao nhất trừ điểm của tỉnh thấp nhất trong cùng vùng. Đây là thước đo độ phân tán đơn giản, phù hợp trực tiếp với biểu đồ Top/Bottom.
- Kết quả mô tả sự khác biệt và cấu trúc của dữ liệu; không đủ để suy luận nguyên nhân hay quan hệ nhân quả.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Đọc đúng nguồn dữ liệu mà dashboard đang sử dụng.
APP_DATA_PATH = Path("../frontend/src/data/data.ts")
raw_text = APP_DATA_PATH.read_text(encoding="utf-8")
json_start = raw_text.index("const data = ") + len("const data = ")
json_end = raw_text.rfind("};") + 1
app_data = json.loads(raw_text[json_start:json_end])

print(f"Đã nạp dữ liệu dashboard từ: {APP_DATA_PATH}")
print(f"Giai đoạn dữ liệu: {app_data['YEARS'][0]}–{app_data['YEARS'][-1]}")

Đã nạp dữ liệu dashboard từ: ../frontend/src/data/data.ts
Giai đoạn dữ liệu: 2022–2026


## 2. Câu hỏi thứ nhất: Điểm trung bình toàn quốc tăng do toàn bộ môn thi hay một số môn?

### Câu hỏi phân tích — What?

Trong giai đoạn 2022–2024, điểm trung bình toàn quốc tăng. Vấn đề cần kiểm tra là mức tăng này có diễn ra tương đối đồng đều ở các môn hay phần lớn được hình thành bởi một nhóm môn cụ thể.

### Ý nghĩa phân tích — Why?

Điểm trung bình toàn quốc là một chỉ số tổng hợp. Khi chỉ số này tăng, người đọc dễ hình thành kết luận rằng kết quả thi được cải thiện trên diện rộng. Tuy nhiên, một trung bình chung có thể che khuất sự trái chiều giữa các môn: một số môn tăng mạnh trong khi các môn khác đi ngang hoặc giảm nhẹ.

Vì vậy, câu hỏi này giúp phân biệt hai tình huống có ý nghĩa rất khác nhau:

- **Cải thiện diện rộng:** phần lớn môn cùng tăng, xu hướng chung có tính nhất quán.
- **Cải thiện tập trung:** chỉ một số môn tăng mạnh và kéo chỉ số chung đi lên, trong khi mặt bằng các môn còn lại ít thay đổi.

### Phương pháp — How?

1. Đọc xu hướng điểm trung bình toàn quốc trong tab **Tổng quan**.
2. Đối chiếu với xu hướng của từng môn trong tab **Xu hướng & Môn học**.
3. Tính thay đổi của từng môn từ năm 2022 đến năm 2024.
4. Kiểm tra tính nhất quán số học giữa mức tăng của chỉ số toàn quốc và trung bình mức thay đổi của chín môn.
5. Tách phần tăng và phần giảm để xác định nhóm môn thực sự tạo nên chuyển động ròng của chỉ số.

### Tiêu chí trả lời

Câu hỏi được xem là có xu hướng **tập trung** nếu số môn tăng chiếm thiểu số, mức tăng giữa các môn có độ chênh lớn, hoặc một nhóm nhỏ môn đóng góp phần lớn thay đổi ròng.


### Bằng chứng trực quan từ ứng dụng

Hai hình dưới đây là bản lưu tĩnh của các biểu đồ trong dashboard, với đúng dữ liệu và trạng thái lọc dùng cho câu hỏi này. Chúng được lưu trong repository nên notebook có thể mở và hiển thị độc lập, không phụ thuộc vào ứng dụng đang chạy.

**Biểu đồ 1 — Tab Tổng quan**  
![Điểm trung bình toàn quốc theo năm — CT2006](assets/q1_national_average_ct2006.svg)

**Biểu đồ 2 — Tab Xu hướng & Môn học**  
![Xu hướng điểm trung bình theo môn — CT2006](assets/q1_subject_trends_ct2006.svg)


In [ ]:
# Phân tích câu hỏi 1: CT2006, giai đoạn 2022–2024.
# Giữ cùng chương trình để tránh trộn thay đổi mặt bằng điểm với thay đổi chương trình thi.
subject_year = pd.DataFrame(app_data["subjectYearMatrix"])
subject_year = subject_year[
    subject_year["program"].eq("CT2006")
    & subject_year["year"].isin([2022, 2023, 2024])
].copy()

subject_change = (
    subject_year
    .pivot(index="subjectName", columns="year", values="average")
    .rename_axis(None, axis=1)
    .rename(columns={2022: "Điểm TB 2022", 2024: "Điểm TB 2024"})
)
subject_change["Mức thay đổi"] = subject_change["Điểm TB 2024"] - subject_change["Điểm TB 2022"]
subject_change["Chiều thay đổi"] = np.where(subject_change["Mức thay đổi"] > 0, "Tăng", "Giảm/không tăng")
subject_change = subject_change.sort_values("Mức thay đổi", ascending=False)

national_change = (
    pd.DataFrame(app_data["nationalAverageByYear"])
    .query("program == 'CT2006' and year in [2022, 2023, 2024]")
    .set_index("year")["value"]
)
national_delta = float(national_change.loc[2024] - national_change.loc[2022])
mean_subject_delta = float(subject_change["Mức thay đổi"].mean())

assert np.isclose(national_delta, mean_subject_delta, atol=0.01)
assert int((subject_change["Mức thay đổi"] > 0).sum()) == 6
assert int((subject_change["Mức thay đổi"] <= 0).sum()) == 3

print("Điểm TB toàn quốc:", {int(year): round(value, 2) for year, value in national_change.items()})
print(f"Thay đổi toàn quốc 2022→2024: {national_delta:+.2f} điểm")
print(f"Mức thay đổi trung bình của 9 môn: {mean_subject_delta:+.2f} điểm")
positive_movement = float(subject_change.loc[subject_change["Mức thay đổi"] > 0, "Mức thay đổi"].sum())
negative_offset = float(subject_change.loc[subject_change["Mức thay đổi"] <= 0, "Mức thay đổi"].sum())
net_movement = float(subject_change["Mức thay đổi"].sum())
top3_movement = float(subject_change.head(3)["Mức thay đổi"].sum())
print(f"Tổng mức tăng của 6 môn: +{positive_movement:.2f}; phần giảm bù trừ của 3 môn: {negative_offset:+.2f}")
print(f"Ba môn tăng mạnh nhất đóng góp {top3_movement:.2f}/{net_movement:.2f} = {top3_movement / net_movement:.1%} thay đổi ròng")
display(subject_change.round(2))

Điểm TB toàn quốc: {2022: 6.4, 2023: 6.53, 2024: 6.75}
Thay đổi toàn quốc 2022→2024: +0.35 điểm
Mức thay đổi trung bình của 9 môn: +0.35 điểm
Tổng mức tăng của 6 môn: +3.20; phần giảm bù trừ của 3 môn: -0.09
Ba môn tăng mạnh nhất đóng góp 2.49/3.11 = 80.1% thay đổi ròng

             Điểm TB 2022  Điểm TB 2024  Mức thay đổi Chiều thay đổi
Sinh học             5.02          6.28          1.26          Tăng
Ngữ văn              6.51          7.23          0.72          Tăng
Địa lý               6.68          7.19          0.51          Tăng
Tiếng Anh            5.16          5.51          0.35          Tăng
Lịch sử              6.34          6.57          0.23          Tăng
GDCD                 8.03          8.16          0.13          Tăng
Toán                 6.47          6.45         -0.02          Giảm/không tăng
Hóa học              6.70          6.68         -0.02          Giảm/không tăng
Vật lý               6.72          6.67         -0.05          Giảm/không tăng

### Kết quả phân tích

#### 1. Chỉ số toàn quốc tăng, nhưng mức tăng giữa các môn phân hóa rõ

Điểm trung bình toàn quốc tăng từ **6,40 điểm năm 2022** lên **6,75 điểm năm 2024**, tương đương **+0,35 điểm**. Dù vậy, chín môn không vận động theo cùng một hướng:

- **Sinh học** tăng mạnh nhất, **+1,26 điểm**.
- **Ngữ văn** tăng **+0,72 điểm** và **Địa lý** tăng **+0,51 điểm**.
- **Tiếng Anh** tăng **+0,35 điểm**, bằng đúng mức tăng của chỉ số toàn quốc.
- **Lịch sử** và **GDCD** tăng lần lượt **+0,23** và **+0,13 điểm**.
- Ngược lại, **Toán, Hóa học và Vật lý** đều giảm nhẹ, lần lượt khoảng **−0,02, −0,02 và −0,05 điểm**.

Như vậy, có **6/9 môn tăng** và **3/9 môn không tăng**. Khoảng biến thiên giữa môn tăng nhiều nhất và môn giảm nhiều nhất là **1,31 điểm**, cho thấy mức tăng chung không đại diện đồng đều cho từng môn.

#### 2. Phần lớn chuyển động ròng tập trung ở ba môn

Nếu cộng các mức thay đổi trước khi chia cho chín môn, sáu môn tăng tạo ra tổng mức tăng **+3,20 điểm**, trong khi ba môn giảm tạo ra phần bù trừ **−0,09 điểm**. Kết quả ròng là **+3,11 điểm**, tương ứng với mức tăng trung bình **+0,35 điểm** của chỉ số toàn quốc.

Riêng **Sinh học, Ngữ văn và Địa lý** tạo ra **+2,49/3,11 điểm**, tương đương khoảng **80,1% thay đổi ròng**. Đây là một phân rã số học, không phải bằng chứng về quan hệ nhân quả; tuy nhiên, nó cho thấy rất rõ mức độ tập trung của xu hướng tăng.

### Trả lời câu hỏi

**Sự gia tăng điểm trung bình toàn quốc trong giai đoạn 2022–2024 chủ yếu đến từ một số môn, không phải từ sự cải thiện đồng đều của toàn bộ môn thi.** Ba động lực nổi bật là Sinh học, Ngữ văn và Địa lý; trong khi Toán, Hóa học và Vật lý gần như ổn định hoặc giảm nhẹ.

Phát hiện này tạo ra một điểm cần thận trọng khi đọc báo cáo tổng hợp: **một chỉ số toàn quốc có thể tăng rõ rệt ngay cả khi một phần đáng kể các môn không cải thiện**. Do đó, đánh giá xu hướng nên luôn đặt điểm trung bình toàn quốc cạnh chuỗi điểm của từng môn thay vì chỉ dựa vào một đường trung bình chung.


## 3. Câu hỏi thứ hai: Chênh lệch vùng miền đến từ toàn bộ địa phương hay một số tỉnh, thành?

### Câu hỏi phân tích — What?

Khi một vùng có điểm trung bình cao hơn vùng khác, chênh lệch đó phản ánh sự khác biệt tương đối đồng đều giữa các địa phương hay bị chi phối bởi một vài tỉnh dẫn đầu? Nói cách khác, cần phân biệt **chênh lệch giữa các vùng** với **phân hóa bên trong từng vùng**.

### Ý nghĩa phân tích — Why?

Điểm trung bình vùng là một phép tổng hợp hữu ích để so sánh mặt bằng, nhưng nó có thể che khuất độ phân tán nội vùng. Hai vùng có điểm trung bình tương tự nhau có thể có cấu trúc hoàn toàn khác: một vùng gồm các tỉnh khá đồng đều, còn vùng kia có khoảng cách lớn giữa nhóm dẫn đầu và nhóm thấp hơn.

Nếu chỉ đọc thứ hạng vùng, người phân tích có thể nhầm lẫn giữa hai kết luận:

- một vùng có mặt bằng chung cao;
- mọi địa phương trong vùng đều có kết quả cao và tương đối đồng nhất.

Hai mệnh đề này không tương đương.

### Phương pháp — How?

1. Giữ bộ lọc của tab **Địa phương & Vùng miền** ở năm 2026, môn **Tổng hợp** và phạm vi **Toàn quốc**.
2. Đọc đồng thời biểu đồ **Điểm TB theo vùng miền** và biểu đồ **Top/Bottom tỉnh, thành theo điểm TB**.
3. Với mỗi vùng, ghi nhận điểm trung bình vùng, tỉnh cao nhất, tỉnh thấp nhất và khoảng cách nội vùng.
4. So sánh lợi thế của tỉnh dẫn đầu so với mức trung bình vùng.
5. Đối chiếu khoảng cách lớn nhất giữa các vùng với khoảng cách lớn nhất giữa các tỉnh trong cùng một vùng.

### Tiêu chí trả lời

Nếu khoảng cách nội vùng nhỏ và tỉnh dẫn đầu chỉ cao hơn nhẹ so với mức vùng, điểm trung bình vùng có thể được xem là đại diện tương đối tốt cho mặt bằng địa phương. Ngược lại, khoảng cách nội vùng lớn cho thấy chỉ số vùng cần được đọc cùng bảng xếp hạng địa phương.


### Bằng chứng trực quan từ ứng dụng

Hình dưới đây là bản lưu tĩnh của tab **Địa phương & Vùng miền** với đúng trạng thái phân tích: năm `2026`, môn `Tổng hợp`, phạm vi `Toàn quốc`, xếp hạng `Cao nhất`, Top `10`. Biểu đồ đặt cạnh nhau hai lớp thông tin: nhóm tỉnh dẫn đầu và mặt bằng điểm trung bình của sáu vùng.

![Địa phương và vùng miền — năm 2026, môn Tổng hợp](assets/q2_region_snapshot_2026.svg)


In [ ]:
# Phân tích câu hỏi 2: tái tạo đúng logic mặc định của RegionTab.
# Bộ lọc: năm 2026, môn Tổng hợp, toàn quốc.
subjects_2026 = {
    subject["id"] for subject in app_data["SUBJECTS"]
    if "CT2018" in subject.get("programs", [])
}

province_rows = pd.DataFrame(app_data["provinceRankings"])
province_rows = province_rows[
    province_rows["year"].eq(2026)
    & province_rows["subjectId"].isin(subjects_2026)
].copy()

# RegionTab lấy trung bình các điểm TB theo môn của từng tỉnh.
province_aggregate = (
    province_rows
    .groupby(["province", "regionId", "regionName"], as_index=False)
    .agg(**{"Điểm TB tỉnh": ("average", "mean")})
)

region_rows = pd.DataFrame(app_data["regionAverages"])
region_rows = region_rows[
    region_rows["year"].eq(2026)
    & region_rows["subjectId"].isin(subjects_2026)
].copy()
region_average = (
    region_rows
    .groupby(["regionId", "regionName"], as_index=False)
    .agg(**{"Điểm TB vùng": ("average", "mean")})
)

rows = []
for _, region in region_average.iterrows():
    local = province_aggregate[province_aggregate["regionId"].eq(region["regionId"])].copy()
    local = local.sort_values("Điểm TB tỉnh", ascending=False)
    top = local.iloc[0]
    bottom = local.iloc[-1]
    rows.append({
        "Vùng": region["regionName"],
        "Điểm TB vùng": region["Điểm TB vùng"],
        "Tỉnh cao nhất": top["province"],
        "Điểm cao nhất": top["Điểm TB tỉnh"],
        "Tỉnh thấp nhất": bottom["province"],
        "Điểm thấp nhất": bottom["Điểm TB tỉnh"],
        "Khoảng cách nội vùng": top["Điểm TB tỉnh"] - bottom["Điểm TB tỉnh"],
        "Lợi thế của tỉnh dẫn đầu": top["Điểm TB tỉnh"] - region["Điểm TB vùng"],
    })

region_comparison = pd.DataFrame(rows).sort_values("Điểm TB vùng", ascending=False)
region_average_gap = float(region_comparison["Điểm TB vùng"].max() - region_comparison["Điểm TB vùng"].min())
largest_within_region_gap = float(region_comparison["Khoảng cách nội vùng"].max())
print(f"Khoảng cách giữa vùng cao nhất và thấp nhất: {region_average_gap:.2f} điểm")
print(f"Khoảng cách nội vùng lớn nhất: {largest_within_region_gap:.2f} điểm ({largest_within_region_gap / region_average_gap:.2f} lần khoảng cách giữa các vùng)")
display(region_comparison.round(2))

Khoảng cách giữa vùng cao nhất và thấp nhất: 0.58 điểm
Khoảng cách nội vùng lớn nhất: 0.94 điểm (1.62 lần khoảng cách giữa các vùng)

                                   Vùng  Điểm TB vùng Tỉnh cao nhất  Điểm cao nhất Tỉnh thấp nhất  Điểm thấp nhất  Khoảng cách nội vùng  Lợi thế của tỉnh dẫn đầu
                     Đồng bằng sông Hồng          5.99       Ninh Bình          6.15       Hưng Yên            5.83                 0.32                      0.16
                           Đông Nam Bộ          5.81       Hồ Chí Minh          5.97       Tây Ninh            5.49                 0.48                      0.16
Bắc Trung Bộ và Duyên hải miền Trung          5.75          Hà Tĩnh          6.17       Khánh Hòa            5.49                 0.68                      0.42
           Trung du và miền núi phía Bắc          5.52          Phú Thọ          5.97       Sơn La            5.03                 0.94                      0.45
                              Tây Nguyên          5.48 

### Kết quả phân tích

#### 1. Mặt bằng vùng cao nhất tương đối đồng đều

Đồng bằng sông Hồng có điểm trung bình vùng cao nhất, **5,99 điểm**. Tuy nhiên, khoảng cách giữa tỉnh cao nhất là Ninh Bình (**6,15 điểm**) và tỉnh thấp nhất là Hưng Yên (**5,83 điểm**) chỉ **0,32 điểm**. Ninh Bình cao hơn mức trung bình vùng **0,16 điểm**, một mức chênh không đủ lớn để cho thấy toàn bộ kết quả của vùng bị quyết định bởi một địa phương đơn lẻ.

Kết quả này cho phép diễn giải thận trọng rằng vị trí dẫn đầu của Đồng bằng sông Hồng gắn với một mặt bằng tương đối đồng đều giữa các tỉnh trong vùng, thay vì chỉ phản ánh một “điểm sáng” riêng lẻ.

#### 2. Một số vùng có phân hóa nội vùng đáng kể

Trung du và miền núi phía Bắc có điểm trung bình vùng **5,52 điểm**, thấp hơn Đồng bằng sông Hồng **0,47 điểm**, nhưng khoảng cách nội vùng lớn nhất: **0,94 điểm**. Phú Thọ đạt **5,97 điểm**, trong khi Sơn La đạt **5,03 điểm**; tỉnh dẫn đầu cao hơn mức vùng **0,45 điểm**.

Bắc Trung Bộ và Duyên hải miền Trung cũng cho thấy mức phân hóa đáng chú ý: khoảng cách nội vùng là **0,68 điểm**, còn Hà Tĩnh cao hơn mức trung bình vùng **0,42 điểm**. Hai trường hợp này cho thấy một điểm trung bình vùng vừa có thể phản ánh mặt bằng chung, vừa có thể che khuất những khác biệt đáng kể giữa các địa phương.

#### 3. Nghịch lý giữa chênh lệch vùng và chênh lệch nội vùng

Khoảng cách giữa vùng cao nhất và vùng thấp nhất chỉ **0,58 điểm** — từ **5,99 điểm** ở Đồng bằng sông Hồng đến **5,41 điểm** ở Đồng bằng sông Cửu Long. Trong khi đó, khoảng cách nội vùng lớn nhất đạt **0,94 điểm**, lớn hơn khoảng **1,62 lần** toàn bộ khoảng cách giữa các vùng.

Đây là kết quả quan trọng nhất của câu hỏi: **khác biệt giữa các tỉnh trong cùng một vùng có thể lớn hơn khác biệt giữa các vùng**. Vì vậy, thứ hạng vùng không đủ để mô tả đầy đủ cấu trúc địa phương.

### Trả lời câu hỏi

**Chênh lệch vùng miền không xuất phát từ một cơ chế duy nhất. Đồng bằng sông Hồng có mặt bằng cao và tương đối đồng đều, trong khi Trung du và miền núi phía Bắc cùng Bắc Trung Bộ và Duyên hải miền Trung có mức phân hóa nội vùng rõ hơn.**

Do đó, không nên suy luận rằng một vùng có điểm trung bình cao thì mọi tỉnh trong vùng đều có kết quả tương đương. Cách đọc phù hợp là kết hợp ba lớp thông tin: điểm trung bình vùng để xác định mặt bằng, Top/Bottom tỉnh để nhận diện cực trị và khoảng cách nội vùng để đánh giá mức độ đồng đều.


## 4. Kết luận và giới hạn diễn giải

### Kết luận chính

Hai câu hỏi cho thấy cùng một nguyên tắc phân tích: **chỉ số tổng hợp cần được giải thích bằng cấu phần tạo nên nó**.

1. Điểm trung bình toàn quốc tăng **+0,35 điểm** trong giai đoạn 2022–2024, nhưng **80,1% thay đổi ròng** đến từ Sinh học, Ngữ văn và Địa lý. Xu hướng toàn quốc vì thế là một kết quả tổng hợp có mức độ tập trung cao, không phải sự cải thiện đồng đều của tất cả môn thi.
2. Chênh lệch giữa vùng cao nhất và thấp nhất là **0,58 điểm**, nhỏ hơn khoảng cách nội vùng lớn nhất **0,94 điểm**. Điều này cho thấy sự khác biệt giữa các tỉnh trong cùng vùng có thể quan trọng hơn thứ hạng giữa các vùng.

### Hàm ý khi sử dụng dashboard

- Khi đánh giá xu hướng theo thời gian, cần đọc đồng thời chỉ số toàn quốc và chuỗi theo môn.
- Khi so sánh địa phương, cần xem điểm trung bình vùng cùng với Top/Bottom tỉnh và độ rộng khoảng cách nội vùng.
- Các tỷ lệ và mức đóng góp trong báo cáo là kết quả phân rã số học của chỉ số tổng hợp; chúng không phải ước lượng tác động nhân quả.

### Giới hạn

- Câu hỏi 1 chỉ sử dụng CT2006 trong giai đoạn 2022–2024 để bảo đảm khả năng so sánh giữa các năm. Không nên suy rộng trực tiếp kết luận này sang CT2018.
- Câu hỏi 2 sử dụng năm 2026 và môn Tổng hợp theo đúng trạng thái phân tích đã chọn. Kết luận có thể thay đổi khi thay đổi năm, môn học hoặc phạm vi địa lý.
- Khoảng cách nội vùng được đo bằng chênh lệch giữa tỉnh cao nhất và thấp nhất. Đây là chỉ báo dễ diễn giải nhưng không thay thế cho các thước đo phân tán đầy đủ như độ lệch chuẩn hoặc khoảng tứ phân vị.
- Dữ liệu cho biết mức độ và cấu trúc khác biệt, nhưng không xác định nguyên nhân của chênh lệch giữa môn học, vùng hoặc tỉnh, thành.
